In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

class SUIMDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))
        self.mask_paths  = glob.glob(os.path.join(mask_dir, "*.png"))

        if len(self.image_paths) == 0:
            self.image_paths = glob.glob(os.path.join(image_dir, "*.png"))
        if len(self.mask_paths) == 0:
            self.mask_paths = glob.glob(os.path.join(mask_dir, "*.jpg"))

        self.image_paths.sort()
        self.mask_paths.sort()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask  = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)
        return image, mask

image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

# Dataset structure: path/dataset/images and path/dataset/masks
base_dir  = os.path.join(path, "dataset")
image_dir = os.path.join(base_dir, "images")
mask_dir  = os.path.join(base_dir, "masks")

full_dataset = SUIMDataset(image_dir, mask_dir,
                           transform=image_transforms,
                           target_transform=mask_transforms)

print("Total Samples:", len(full_dataset))
assert len(full_dataset) > 0, "Dataset is empty. Check dataset/images and dataset/masks paths."

# Train/Validation split (80/20)
num_samples = len(full_dataset)
indices = torch.randperm(num_samples).tolist()
split = int(0.8 * num_samples)
train_idx, val_idx = indices[:split], indices[split:]

train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training Samples: {len(train_dataset)}, Validation Samples: {len(val_dataset)}")

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

# Display some images and masks
for i in range(3):
    img, mask = full_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO

In [ ]:
# TO DO
!pip install -q segmentation_models_pytorch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
).to(device)


In [ ]:
# TO DO

import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images = images.to(device)
        masks  = masks.to(device).squeeze(1).long()

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    for images, masks in dataloader:
        images = images.to(device)
        masks  = masks.to(device).squeeze(1).long()

        outputs = model(images)
        loss = criterion(outputs, masks)

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# TO DO
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 2
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

plt.plot(range(1, num_epochs+1), train_losses, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses, marker='o', label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()


In [ ]:
# TO DO
import random
import torch.nn.functional as F

model.eval()
samples = random.sample(range(len(val_dataset)), 5)

for idx in samples:
    img, mask = val_dataset[idx]

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred_mask = torch.argmax(F.softmax(pred, dim=1), dim=1).cpu().squeeze()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")
    plt.show()